### Limpieza del dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [2]:
def limpiar_dataset(ruta_archivo):
    # 1. Cargar el dataset con el separador correcto (;)
    df = pd.read_csv(ruta_archivo, sep=';')
    
    # 2. Reemplazar el caracter '-' por NaN para facilitar la limpieza
    df = df.replace('-', np.nan)
    
    # 3. Limpiar columnas de Diagnóstico y Procedimiento
    # Extraer solo el código (lo que está antes del ' - ')
    cols_medicas = [c for c in df.columns if 'Diag' in c or 'Proced' in c]
    for col in cols_medicas:
        df[col] = df[col].str.split(' - ').str[0]
        df[col] = df[col].replace('nan', 'EMPTY')
    
    # 4. Convertir Sexo a numérico
    df["Sexo_Binario"] = df["Sexo (Desc)"].map({"Mujer": 0, "Hombre": 1})
    
    # 5. Normalizar la Edad
    scaler = MinMaxScaler()
    df['Edad en años'] = scaler.fit_transform(df[['Edad en años']])
    
    # 6. Preparar la columna objetivo (GRD)
    # Extraer solo el código del GRD (ej: 184103)
    df['GRD_Cod'] = df['GRD'].str.split(' - ').str[0]
    
    # Rellenar los NaN restantes con un string vacío o token 'vacío'
    df = df.fillna('None')
    
    return df

# Uso del script
df_limpio = limpiar_dataset('data/dataset_elpino.csv')
print(df_limpio.head())


  Diag 01 Principal (cod+des) Diag 02 Secundario (cod+des)  \
0                       A41.8                        B37.6   
1                       U07.1                        J12.8   
2                       K56.5                        R57.2   
3                       K76.8                        K66.1   
4                       T81.0                        Y83.2   

  Diag 03 Secundario (cod+des) Diag 04 Secundario (cod+des)  \
0                        I39.8                          N10   
1                        R06.0                          R05   
2                        R57.1                          J80   
3                        N18.5                        D64.9   
4                        S31.1                       S36.80   

  Diag 05 Secundario (cod+des) Diag 06 Secundario (cod+des)  \
0                        B96.1                        L89.9   
1                        R50.9                        Z29.0   
2                          Y95                        J15.0

In [3]:
# Guardar el DataFrame limpio en un nuevo archivo CSV
df_limpio.to_csv('dataset_elpino_LIMPIO.csv', index=False, sep=';', encoding='utf-8-sig')

print("¡Copia guardada con éxito!")

¡Copia guardada con éxito!


### Transformación para la Red Neuronal

In [4]:
# Aislar las columnas médicas
cols_medicas = [c for c in df_limpio.columns if 'Diag' in c or 'Proced' in c]

# Crear un diccionario (LabelEncoder) global para todos los códigos médicos
codificador_global = LabelEncoder()
todos_los_codigos = pd.concat([df_limpio[col] for col in cols_medicas]).astype(str).unique()
codificador_global.fit(todos_los_codigos)

# Convertir el texto de los diagnósticos a números enteros
for col in cols_medicas:
    df_limpio[col] = codificador_global.transform(df_limpio[col].astype(str))

# Preparar la variable Y (Lo que predecimos: El GRD)
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(df_limpio['GRD_Cod'].astype(str))
y_categorical = to_categorical(y_encoded) # Formato requerido por redes multicapa
num_clases = y_categorical.shape[1]

# Preparar la variable X (Las características: Edad, Sexo y Diagnósticos)
features = ['Edad en años', 'Sexo_Binario'] + cols_medicas
X = df_limpio[features].values

# Separar datos para Entrenar (80%) y para Testear (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

print(f"Datos listos. Variables de entrada: {X_train.shape[1]}, Clases a predecir: {num_clases}")

Datos listos. Variables de entrada: 67, Clases a predecir: 526


In [5]:
def crear_red_neuronal(capas_ocultas):
    """
    Función para crear la red neuronal dinámicamente.
    Ejemplo: capas_ocultas=[128, 64] crea dos capas con esa cantidad de perceptrones.
    """
    modelo = Sequential()
    
    # Capa de entrada y primera capa oculta
    modelo.add(Dense(capas_ocultas[0], input_dim=X_train.shape[1], activation='relu'))
    
    # Agregar las demás capas ocultas que definas
    for perceptrones in capas_ocultas[1:]:
        modelo.add(Dense(perceptrones, activation='relu'))
        modelo.add(Dropout(0.2)) # Apaga neuronas al azar para evitar sobreajuste
        
    # Capa de salida (Múltiples clases = softmax)
    modelo.add(Dense(num_clases, activation='softmax'))
    
    # Compilar el modelo
    modelo.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    return modelo

In [6]:
# Define las estructuras que quieres evaluar: [Capa1, Capa2, Capa3...]
experimentos = [
    [32, 16],          # Experimento 1: 2 capas pequeñas
    [128, 64, 32],     # Experimento 2: 3 capas (más profunda)
    [256, 128]         # Experimento 3: 2 capas (más ancha)
]

for perceptrones_por_capa in experimentos:
    print(f"\n--- Evaluando Red con Arquitectura: {perceptrones_por_capa} ---")
    
    # 1. Crear el modelo con los perceptrones definidos
    mi_red = crear_red_neuronal(perceptrones_por_capa)
    
    # 2. Entrenar el modelo (epochs=10 para no tardar mucho, puedes subirlo a 50 o 100)
    mi_red.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0, validation_split=0.1)
    
    # 3. Evaluar con los datos de prueba (test)
    perdida, precision = mi_red.evaluate(X_test, y_test, verbose=0)
    
    print(f"Precisión (Accuracy): {precision:.4f} ({(precision*100):.2f}%)")


--- Evaluando Red con Arquitectura: [32, 16] ---


c:\Users\nicol\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Precisión (Accuracy): 0.0467 (4.67%)

--- Evaluando Red con Arquitectura: [128, 64, 32] ---
Precisión (Accuracy): 0.0467 (4.67%)

--- Evaluando Red con Arquitectura: [256, 128] ---
Precisión (Accuracy): 0.0467 (4.67%)
